In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [2]:
load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

CITY_COUNTRY = {"Amsterdam": "NLD", "Barcelona": "ESP", "Berlin": "DEU", "Columbus": "USA"}

In [3]:
tables = ["raw_listings", "raw_calendar", "raw_weather", "raw_economic_indicators", "raw_landmarks"]
for t in tables:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"][0]
    print(f"{t:28s} {n:>12,}")

print()
print(pd.read_sql("SELECT city, COUNT(*) AS n FROM raw_listings GROUP BY city", engine))

raw_listings                       41,183
raw_calendar                   15,032,960
raw_weather                         1,472
raw_economic_indicators                96
raw_landmarks                       1,640

        city      n
0  Amsterdam  10612
1  Barcelona  15236
2     Berlin  12376
3   Columbus   2959


In [5]:
listings_raw = pd.read_sql("SELECT * FROM raw_listings", engine)
print(listings_raw.shape)
listings_raw.head()

(41183, 91)


,city,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,Columbus,90676,https://www.airbnb.com/rooms/90676,20260922161657,2026-09-25,None,Short North - Italianate Cottage,Just steps from High Street and all the action...,None,https://a0.muscache.com/pictures/950e43cd-53f3...,...,4.86,4.92,4.77,STRP26000269,None,3,3,0,0,5.31
1,Columbus,591101,https://www.airbnb.com/rooms/591101,20260922161657,2026-09-25,None,Bellows Studio Loft Apartment,Famous American artist George Bellows home wit...,None,https://a0.muscache.com/pictures/32b28442-ddf3...,...,4.9,4.89,4.89,2019-1230,None,1,0,1,0,2.15
2,Columbus,927867,https://www.airbnb.com/rooms/927867,20260922161657,2026-09-25,None,Full Private Room at the Hostel,The Wayfaring Buckeye Hostel is a social place...,None,https://a0.muscache.com/pictures/08033ebe-286c...,...,4.86,4.65,4.67,STRP26000385,None,4,0,4,0,0.54
3,Columbus,1183297,https://www.airbnb.com/rooms/1183297,20260922161657,2026-09-25,None,Hannah's Haus Historic German Village 3BR+Parking,"Experience Hannah's Haus, a stunning 3-bedroom...",None,https://a0.muscache.com/pictures/3a8c9d7f-d77d...,...,4.98,4.78,4.82,None,None,4,4,0,0,1.47
4,Columbus,1217678,https://www.airbnb.com/rooms/1217678,20260922161657,2026-09-25,None,Comfortable rooms in Clintonville 1,"A cozy, warm, inviting place to stay in the he...",None,https://a0.muscache.com/pictures/airflow/Hosti...,...,4.99,4.97,4.94,2025-2824,None,2,0,2,0,1.94


In [6]:
print(listings_raw.groupby("city")["price"].apply(lambda s: s.dropna().head(5).tolist()))

city
Amsterdam    [$97.00, $424.29, $225.00, $187.91, $273.00]
Barcelona    [$274.00, $407.67, $545.60, $44.47, $187.46]
Berlin          [$97.00, $69.60, $219.00, $65.67, $74.00]
Columbus      [$208.00, $126.50, $77.00, $160.88, $70.07]
Name: price, dtype: object


In [7]:
tmp = listings_raw.replace(r"^\s*$", np.nan, regex=True)
missing = tmp.isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3).head(40)

host_response_rate              1.000
host_neighbourhood              1.000
host_thumbnail_url              1.000
host_since                      1.000
host_total_listings_count       1.000
host_verifications              1.000
host_acceptance_rate            1.000
neighbourhood                   1.000
calendar_updated                1.000
neighborhood_overview           1.000
bathrooms                       1.000
host_response_time              1.000
instant_bookable                1.000
source                          0.642
host_about                      0.405
neighbourhood_group_cleansed    0.330
price_quote_price_per_night     0.292
price_quote_total_price         0.292
price                           0.285
estimated_revenue_l365d         0.285
license                         0.258
price_quote_checkout_date       0.250
price_quote_checkin_date        0.250
price_quote_raw                 0.250
host_location                   0.194
review_scores_accuracy          0.167
review_score

In [8]:
tmp.columns[tmp.isna().all()].tolist()

['neighborhood_overview',
 'host_since',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_thumbnail_url',
 'host_neighbourhood',
 'host_total_listings_count',
 'host_verifications',
 'neighbourhood',
 'bathrooms',
 'calendar_updated',
 'instant_bookable']

In [10]:
def money(s):
    """'$1,200.00' -> 1200.0"""
    return pd.to_numeric(
        s.astype(str).str.replace(r"[^\d.]", "", regex=True).replace("", np.nan),
        errors="coerce")

inv = listings_raw[["city", "has_availability", "price", "price_quote_price_per_night"]].copy()
inv["price_num"] = money(inv["price"])
inv["quote_num"] = money(inv["price_quote_price_per_night"])

print(inv.groupby("city").agg(
    rows=("price_num", "size"),
    price_missing=("price_num", lambda s: round(s.isna().mean(), 3)),
    quote_missing=("quote_num", lambda s: round(s.isna().mean(), 3))))

print("\nprice missing but quote available:", (inv.price_num.isna() & inv.quote_num.notna()).sum())
print("\nshare of missing price by has_availability:")
print(pd.crosstab(inv.has_availability, inv.price_num.isna(), normalize="index").round(3))

            rows  price_missing  quote_missing
city                                          
Amsterdam  10612          0.431          0.431
Barcelona  15236          0.172          0.183
Berlin     12376          0.341          0.350
Columbus    2959          0.110          0.110

price missing but quote available: 0

share of missing price by has_availability:
price_num         False  True 
has_availability              
f                 0.896  0.104
t                 0.707  0.293


In [11]:
listings = listings_raw.replace(r"^\s*$", np.nan, regex=True)

all_empty = listings.columns[listings.isna().all()].tolist()

USELESS = ["listing_url", "scrape_id", "source", "picture_url", "host_url", "host_profile_url",
           "host_thumbnail_url", "host_picture_url", "host_about", "description",
           "neighborhood_overview", "host_profile_id",
           "price_quote_checkin_date", "price_quote_checkout_date",
           "price_quote_total_price", "price_quote_raw"]

listings = listings.drop(columns=[c for c in set(all_empty + USELESS) if c in listings.columns])
print(listings.shape)

(41183, 64)


In [12]:
NUM = ["accommodates", "bedrooms", "beds", "minimum_nights", "maximum_nights",
       "minimum_minimum_nights", "maximum_minimum_nights", "minimum_maximum_nights",
       "maximum_maximum_nights", "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",
       "availability_30", "availability_60", "availability_90", "availability_365", "availability_eoy",
       "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
       "estimated_occupancy_l365d", "estimated_revenue_l365d", "reviews_per_month",
       "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
       "review_scores_checkin", "review_scores_communication", "review_scores_location",
       "review_scores_value", "latitude", "longitude", "host_listings_count",
       "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
       "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
       "hosts_time_as_user_years", "hosts_time_as_user_months",
       "hosts_time_as_host_years", "hosts_time_as_host_months"]
for c in NUM:
    if c in listings.columns:
        listings[c] = pd.to_numeric(listings[c], errors="coerce")

listings["price"] = money(listings["price"])
listings = listings.drop(columns=["price_quote_price_per_night"], errors="ignore")

for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "has_availability"]:
    if c in listings.columns:
        listings[c] = listings[c].map({"t": True, "f": False})

for c in ["last_scraped", "calendar_last_scraped", "first_review", "last_review"]:
    if c in listings.columns:
        listings[c] = pd.to_datetime(listings[c], errors="coerce")

for c in ["room_type", "property_type", "neighbourhood_cleansed", "neighbourhood_group_cleansed", "city"]:
    if c in listings.columns:
        listings[c] = listings[c].astype("string").str.strip()

print(listings.dtypes.value_counts())

float64           26
int64             16
object             9
string[python]     5
datetime64[ns]     4
bool               3
Name: count, dtype: int64


In [13]:
bt = listings["bathrooms_text"].astype("string").str.lower()
num = pd.to_numeric(bt.str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")
listings["bathrooms"] = num.where(num.notna(), np.where(bt.str.contains("half", na=False), 0.5, np.nan))
listings["bathroom_shared"] = bt.str.contains("shared", na=False)

listings["price_missing"] = listings["price"].isna()
listings["has_reviews"] = listings["number_of_reviews"].fillna(0) > 0
listings["has_license"] = listings["license"].notna()
listings["host_tenure_years"] = (listings["hosts_time_as_host_years"].fillna(0)
                                 + listings["hosts_time_as_host_months"].fillna(0) / 12)
listings = listings.drop(columns=["license"])

print(listings[["bathrooms", "bathroom_shared", "price_missing", "has_reviews",
                "has_license", "host_tenure_years"]].describe(include="all"))

        bathrooms bathroom_shared price_missing has_reviews has_license  \
count     41157.0           41183         41183       41183       41183   
unique       <NA>               2             2           2           2   
top          <NA>           False         False        True        True   
freq         <NA>           35391         29440       34298       30559   
mean     1.292344             NaN           NaN         NaN         NaN   
std       0.72062             NaN           NaN         NaN         NaN   
min           0.0             NaN           NaN         NaN         NaN   
25%           1.0             NaN           NaN         NaN         NaN   
50%           1.0             NaN           NaN         NaN         NaN   
75%           1.5             NaN           NaN         NaN         NaN   
max          17.0             NaN           NaN         NaN         NaN   

        host_tenure_years  
count        41183.000000  
unique                NaN  
top            

In [14]:
before = len(listings)
listings = listings.drop_duplicates(subset=["city", "id"])
print("Duplicate rows removed:", before - len(listings))

Duplicate rows removed: 0


In [15]:
bad_price = (listings["price"] <= 0).sum()
listings.loc[listings["price"] <= 0, "price"] = np.nan
listings["price_missing"] = listings["price"].isna()
print("bad prices:", bad_price)

bad prices: 0


In [16]:
print("bathrooms 0:", (listings["bathrooms"] == 0).sum(), "| bathrooms 6 or more:", (listings["bathrooms"] >= 6).sum())
print("hosts with tenure = 0:", (listings["host_tenure_years"] == 0).sum())

bathrooms 0: 816 | bathrooms 6 or more: 153
hosts with tenure = 0: 259


In [17]:
print("bathrooms 0: n of guests and room typeF")
print(listings.loc[listings["bathrooms"] == 0, "bathrooms_text"].value_counts(dropna=False).head(5))

print("\nbathrooms 6 or more: n of guests and room type")
big = listings[listings["bathrooms"] >= 6]
print(big[["accommodates", "room_type"]].describe(include="all").loc[["count", "mean", "top"]])

print("\nhosts with tenure = 0: is the source missing?")
zero = listings["host_tenure_years"] == 0
print(listings.loc[zero, ["hosts_time_as_host_years", "hosts_time_as_host_months"]].isna().mean())

bathrooms 0: n of guests and room typeF
bathrooms_text
0 baths           487
0 shared baths    329
Name: count, dtype: int64

bathrooms 6 or more: n of guests and room type
       accommodates        room_type
count     153.00000              153
mean       13.24183              NaN
top             NaN  Entire home/apt

hosts with tenure = 0: is the source missing?
hosts_time_as_host_years     0.015444
hosts_time_as_host_months    0.015444
dtype: float64


In [18]:
def iqr_flag(s):
    q1, q3 = s.quantile([.25, .75])
    return s > q3 + 1.5 * (q3 - q1)

listings["price_outlier"] = listings.groupby("city")["price"].transform(iqr_flag).fillna(False)

In [19]:
listings["min_nights_outlier"] = listings["minimum_nights"] > 365

In [20]:
CENTER = {"Amsterdam": (52.3676, 4.9041), "Barcelona": (41.3874, 2.1686),
          "Berlin": (52.5200, 13.4050), "Columbus": (39.9612, -82.9988)}
lat0 = listings["city"].map(lambda c: CENTER[c][0])
lon0 = listings["city"].map(lambda c: CENTER[c][1])
listings["geo_outlier"] = ((listings.latitude - lat0).abs() > 0.5) | ((listings.longitude - lon0).abs() > 0.5)

In [21]:
summary = pd.DataFrame({
    "raw_rows": listings_raw.groupby("city").size(),
    "clean_rows": listings.groupby("city").size(),
    "with_price": listings[listings.price.notna()].groupby("city").size(),
    "price_outliers": listings.groupby("city")["price_outlier"].sum(),
    "min_nights_outliers": listings.groupby("city")["min_nights_outlier"].sum(),
    "geo_outliers": listings.groupby("city")["geo_outlier"].sum(),
})
print(summary)
print("\ncolumns: before", listings_raw.shape[1], "-> after", listings.shape[1])

           raw_rows  clean_rows  with_price  price_outliers  \
city                                                          
Amsterdam     10612       10612        6039             333   
Barcelona     15236       15236       12610             635   
Berlin        12376       12376        8157             434   
Columbus       2959        2959        2634             146   

           min_nights_outliers  geo_outliers  
city                                          
Amsterdam                    1             0  
Barcelona                    4             0  
Berlin                      10             0  
Columbus                     0             0  

columns: before 91 -> after 71


In [22]:
listings.to_csv(OUT / "listings_clean.csv", index=False)
print("Saved to:", OUT / "listings_clean.csv")

Saved to: ..\data\processed\listings_clean.csv
